# ID Document VLM Fine-Tuning & Robustness Benchmark Pipeline

This notebook demonstrates fine-tuning **Qwen2-VL-2B-Instruct** with **PEFT LoRA** on synthetic document data and evaluating zero-shot baseline vs fine-tuned extraction performance across clean and degraded ID card images.

In [ ]:
# Phase 0: Setup & Dependency Installation
!nvidia-smi
!pip install -q transformers peft accelerate bitsandbytes datasets pillow faker qwen-vl-utils torch pandas matplotlib seaborn Levenshtein tqdm

In [ ]:
# Phase 1: Generate Clean Synthetic ID Documents (800 samples)
!python scripts/generate_synthetic_data.py --count 800 --output_dir data/clean
# Apply Degradation Corruptions (Blur, JPEG, Rotation, Glare, Combined)
!python scripts/degrade_images.py --input_dir data/clean --output_dir data/degraded
# Render QA Grid Sample
!python scripts/visualize_samples.py --clean_dir data/clean --degraded_dir data/degraded --output_path results/sample_qa_grid.png

In [ ]:
# Phase 2: Baseline Zero-Shot Evaluation
!python scripts/eval_harness.py --model_path Qwen/Qwen2-VL-2B-Instruct --output_csv results/baseline_results.csv

In [ ]:
# Phase 3: PEFT LoRA Fine-Tuning on Clean Data (3 Epochs)
!python scripts/finetune_lora.py --model_path Qwen/Qwen2-VL-2B-Instruct --train_dir data/clean --output_dir results/qwen2_vl_lora_checkpoint --epochs 3 --batch_size 2 --grad_accum 4

In [ ]:
# Phase 4: Post-Fine-Tune Evaluation & Visual Comparison
!python scripts/eval_harness.py --model_path Qwen/Qwen2-VL-2B-Instruct --adapter_path results/qwen2_vl_lora_checkpoint --output_csv results/finetuned_results.csv
!python scripts/analyze_results.py --baseline_csv results/baseline_results.csv --finetuned_csv results/finetuned_results.csv --output_chart results/comparison_chart.png